# 07 — CEX.IO Connector Exploration

**Goal**: Explore the CEX.IO public REST API to inform our production connector.

**Scope**:
- Verify Ohio eligibility (CRITICAL gate — per DEC-006)
- Test raw `httpx` approach (no existing async SDK — custom connector per DEC-011)
- Map CEX.IO symbols to canonical pairs
- Parse responses into our `TopOfBook` dataclass
- Document rate limits, error handling, edge cases
- Verify which of our 8 target pairs are available
- Answer key design questions for the production connector

**Target Pairs** (from PROJECT_INSTRUCTIONS.md):
- BTC/USD, BTC/USDC
- LTC/USD, LTC/USDC, LTC/BTC
- SOL/USD, SOL/USDC, SOL/BTC

**Key Differences from Previous Exchanges**:
- CEX.IO uses URL path segments: `/api/ticker/BTC/USD` (slash-separated in URL)
- Response pair format uses colon separator: `BTC:USD`
- Ticker has bid/ask prices but **NO bid/ask sizes** (same as Gemini, Bitstamp)
- Must use order book endpoint for TopOfBook with sizes
- Order book entries are **arrays-of-arrays** `[[price, amount]]` (same as Bitstamp)
- Timestamps: both `timestamp` (Unix seconds integer) and `timestamp_ms` (Unix milliseconds integer)
- Rate limits: 300 req/10 min for public API (= 30 req/min, conservative)
- Two separate API platforms: Exchange API (cex.io/api) and Spot Trading API (trade.cex.io)
- We use the Exchange API public endpoints (no auth needed for Phase 1)

**Lessons Applied** (from LESSONS_LEARNED.md):
- LL-001: Verify exact symbol format, don't assume
- LL-002: Document actual response shapes from live API, not just docs
- LL-003: Test rate limit behavior before building production connector
- LL-010: All prices/sizes via `to_decimal()`, never float
- LL-050: Use `nest_asyncio.apply()` for async in Jupyter (but beware 3.14 issues)
- LL-052: No batch endpoint assumption — verify before building connector
- LL-060: Ticker endpoints often lack bid/ask sizes — verify and use order book if needed
- LL-070: Order book entries may be arrays-of-arrays, not arrays-of-objects

## 1. Ohio Eligibility Verification (CRITICAL GATE)

Per DEC-006, every exchange must have verified Ohio eligibility before proceeding.

### Research Findings

**CEX.IO Ohio MTL: ✅ CONFIRMED — License OHMT176**

Evidence (from https://cex.io/legal-security/us, fetched 2026-02-16):

| Field | Value |
|-------|-------|
| License Type | Money Transmitter License |
| License Number | OHMT176 |
| Regulator | Ohio Division of Financial Institutions |
| Address | 77 South High Street, 21st Floor, Columbus, OH 43215 |
| FinCEN MSB | Registered (NMLS ID: 1804170) |
| Total US MTLs | 36+ states |

**Additional Context**:
- CEX.IO has been operating in the US since 2015 (FinCEN MSB registration)
- The Ohio MTL can be verified via NMLS Consumer Access: https://www.nmlsconsumeraccess.org/EntityDetails.aspx/COMPANY/1804170
- CEX.IO Corp is registered at 900 E Diehl Rd STE 110, Naperville, IL 60563
- Founded 2013; 5M+ registered accounts; $1.6B avg daily spot turnover (CoinGecko April 2025)

**Verdict**: CEX.IO is Ohio-eligible. Proceed with full exploration notebook.

→ **Decision**: Add to DECISION_LOG as DEC-023 (CEX.IO Ohio eligibility confirmed)

## 2. Setup & Imports

In [1]:
# Install dependencies (run once)
# !pip install httpx

In [2]:
import sys
import time
from pprint import pprint

import httpx

sys.path.insert(0, "../src")

# Our existing infrastructure — reuse, don't reimplement
from uscryptoarb.marketdata.topofbook import TopOfBook, tob_from_raw
from uscryptoarb.validation.guards import require_present
from uscryptoarb.venues.symbol_translator import create_translator

BASE_URL = "https://cex.io/api"
print(f"Base URL: {BASE_URL}")
print(f"Python version: {sys.version}")

Base URL: https://cex.io/api
Python version: 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]


## 3. Symbol Discovery & Mapping

CEX.IO uses a unique symbol format:
- **URL path**: `/api/ticker/{symbol1}/{symbol2}` → e.g., `/api/ticker/BTC/USD`
- **Response pair field**: colon-separated → `BTC:USD`

This is different from all other exchanges:
- Kraken: `XXBTZUSD` (legacy format)
- Coinbase: `BTC-USD` (hyphen-separated)
- Gemini/Bitstamp: `btcusd` (lowercase no-separator)
- OKX: `BTC-USDT` (hyphen-separated)

For our connector, we need to map canonical `BTC/USD` → CEX.IO URL segments `BTC/USD`.
Interestingly, our canonical format already uses `/` — but the connector needs to know
that CEX.IO expects the pair as two separate URL path segments, not a single string.

In [3]:
# CEX.IO symbol mapping
# Canonical pair → CEX.IO format (symbol1, symbol2 for URL path)
# CEX.IO uses uppercase, slash in URL: /api/ticker/BTC/USD
# For our SymbolTranslator, we store as "BTC/USD" and split on "/" for URL construction

CEXIO_SYMBOL_MAP: dict[str, str] = {
    "BTC/USD": "BTC/USD",
    "BTC/USDC": "BTC/USDC",
    "LTC/USD": "LTC/USD",
    "LTC/USDC": "LTC/USDC",
    "LTC/BTC": "LTC/BTC",
    "SOL/USD": "SOL/USD",
    "SOL/USDC": "SOL/USDC",
    "SOL/BTC": "SOL/BTC",
}


# Helper to convert canonical pair to URL path segments
def pair_to_url_path(cexio_symbol: str) -> str:
    """Convert 'BTC/USD' to URL path segment 'BTC/USD'."""
    # For CEX.IO, the symbol IS the path segment already
    return cexio_symbol


print("Symbol mapping:")
for canonical, cexio in CEXIO_SYMBOL_MAP.items():
    parts = cexio.split("/")
    print(f"  {canonical:10s} → URL: /api/ticker/{parts[0]}/{parts[1]}")

Symbol mapping:
  BTC/USD    → URL: /api/ticker/BTC/USD
  BTC/USDC   → URL: /api/ticker/BTC/USDC
  LTC/USD    → URL: /api/ticker/LTC/USD
  LTC/USDC   → URL: /api/ticker/LTC/USDC
  LTC/BTC    → URL: /api/ticker/LTC/BTC
  SOL/USD    → URL: /api/ticker/SOL/USD
  SOL/USDC   → URL: /api/ticker/SOL/USDC
  SOL/BTC    → URL: /api/ticker/SOL/BTC


In [4]:
# Verify which pairs are actually available via currency_limits endpoint
# This is a POST endpoint (unusual for public data)
resp = httpx.post(f"{BASE_URL}/currency_limits")
print(f"Status: {resp.status_code}")
limits = resp.json()

if limits.get("ok") == "ok":
    available_pairs = {}
    for pair_info in limits["data"]["pairs"]:
        sym1 = pair_info["symbol1"]
        sym2 = pair_info["symbol2"]
        canonical = f"{sym1}/{sym2}"
        available_pairs[canonical] = pair_info

    print(f"\nTotal available pairs: {len(available_pairs)}")
    print("\nOur target pairs availability:")
    for target in CEXIO_SYMBOL_MAP:
        if target in available_pairs:
            info = available_pairs[target]
            print(
                f"  ✅ {target:10s} pricePrecision={info.get('pricePrecision')} "
                f"minLotSize={info.get('minLotSize')} "
                f"minLotSizeS2={info.get('minLotSizeS2')}"
            )
        else:
            print(f"  ❌ {target:10s} NOT AVAILABLE")
else:
    print(f"Error: {limits}")

Status: 200

Total available pairs: 706

Our target pairs availability:
  ✅ BTC/USD    pricePrecision=1 minLotSize=0.00035981 minLotSizeS2=10
  ❌ BTC/USDC   NOT AVAILABLE
  ✅ LTC/USD    pricePrecision=3 minLotSize=0.13798541 minLotSizeS2=10
  ✅ LTC/USDC   pricePrecision=2 minLotSize=0.15 minLotSizeS2=10
  ✅ LTC/BTC    pricePrecision=8 minLotSize=0.13798541 minLotSizeS2=0.00035981
  ✅ SOL/USD    pricePrecision=4 minLotSize=0.217106 minLotSizeS2=10
  ✅ SOL/USDC   pricePrecision=3 minLotSize=0.01 minLotSizeS2=10
  ❌ SOL/BTC    NOT AVAILABLE


In [5]:
# SymbolTranslator round-trip test
translator = create_translator("cexio", CEXIO_SYMBOL_MAP)

print("SymbolTranslator round-trip:")
for canonical in CEXIO_SYMBOL_MAP:
    cexio_sym = translator.to_venue_symbol(canonical)
    back = translator.to_canonical(cexio_sym)
    status = "✅" if back == canonical else "❌"
    print(f"  {status} {canonical} → {cexio_sym} → {back}")

# USD ≠ USDC verification (DEC-001)
print("\nUSD ≠ USDC verification:")
btc_usd = translator.to_venue_symbol("BTC/USD")
btc_usdc = translator.to_venue_symbol("BTC/USDC")
print(f"  BTC/USD  → {btc_usd}")
print(f"  BTC/USDC → {btc_usdc}")
print(f"  Distinct: {btc_usd != btc_usdc}")

SymbolTranslator round-trip:
  ✅ BTC/USD → BTC/USD → BTC/USD
  ✅ BTC/USDC → BTC/USDC → BTC/USDC
  ✅ LTC/USD → LTC/USD → LTC/USD
  ✅ LTC/USDC → LTC/USDC → LTC/USDC
  ✅ LTC/BTC → LTC/BTC → LTC/BTC
  ✅ SOL/USD → SOL/USD → SOL/USD
  ✅ SOL/USDC → SOL/USDC → SOL/USDC
  ✅ SOL/BTC → SOL/BTC → SOL/BTC

USD ≠ USDC verification:
  BTC/USD  → BTC/USD
  BTC/USDC → BTC/USDC
  Distinct: True


## 4. Ticker Endpoint

CEX.IO provides two ticker endpoints:
- Single: `GET /api/ticker/{symbol1}/{symbol2}`
- Batch: `GET /api/tickers/{symbol1}/{symbol2}/.../{symbolN}`

**Critical Question**: Does the ticker include bid/ask SIZES?

Based on API documentation, the ticker response shape is:
```json
{
  "timestamp": "1513166480",
  "low": "17200",
  "high": "17920",
  "last": "17420",
  "volume": "2042.59581123",
  "volume30d": "81150.14153359",
  "bid": 17400.04,
  "ask": 17418.2
}
```

Note: `bid` and `ask` are **numeric** (not strings), and there are **NO size fields**.
This matches the pattern from Gemini (LL-060) and Bitstamp.

In [6]:
# Test single ticker
resp = httpx.get(f"{BASE_URL}/ticker/BTC/USD")
print(f"Status: {resp.status_code}")
ticker = resp.json()
pprint(ticker)

# Check for bid/ask sizes
print(f"\nFields present: {list(ticker.keys())}")
has_bid_size = any(k in ticker for k in ["bidSize", "bid_size", "bidQty", "bidVolume"])
has_ask_size = any(k in ticker for k in ["askSize", "ask_size", "askQty", "askVolume"])
print(f"Bid size field: {'YES' if has_bid_size else 'NO ⚠️'}")
print(f"Ask size field: {'YES' if has_ask_size else 'NO ⚠️'}")
print("\n→ Ticker lacks bid/ask sizes. Must use order book endpoint (per LL-060).")

Status: 200
{'ask': 67233.5,
 'bid': 67208.5,
 'high': '70092',
 'last': '67233.5',
 'low': '68654.2',
 'pair': 'BTC:USD',
 'priceChange': '-2442.5',
 'priceChangePercentage': '-3.51',
 'timestamp': '1772914204',
 'volume': '0.35023893',
 'volume30d': '0.00000000'}

Fields present: ['timestamp', 'low', 'high', 'last', 'volume', 'volume30d', 'bid', 'ask', 'priceChange', 'priceChangePercentage', 'pair']
Bid size field: NO ⚠️
Ask size field: NO ⚠️

→ Ticker lacks bid/ask sizes. Must use order book endpoint (per LL-060).


In [7]:
# Test batch tickers endpoint
# Format: /api/tickers/{sym1}/{sym2}/...
# Each pair is two path segments, separated by /
# Multiple pairs are just concatenated: /api/tickers/BTC/USD/LTC/USD/SOL/USD
pairs_path = "/".join(
    f"{p.split('/')[0]}/{p.split('/')[1]}" for p in ["BTC/USD", "LTC/USD", "SOL/USD"]
)
resp = httpx.get(f"{BASE_URL}/tickers/{pairs_path}")
print(f"Batch tickers status: {resp.status_code}")
batch = resp.json()

if isinstance(batch, dict) and "data" in batch:
    for t in batch["data"]:
        print(f"  {t.get('pair'):10s} bid={t.get('bid')} ask={t.get('ask')}")
else:
    pprint(batch)

Batch tickers status: 200
  BTC:USD    bid=67208.5 ask=67233.5
  ETH:USD    bid=1962.5 ask=1964
  BCH:USD    bid=445.89 ask=448.31
  DASH:USD   bid=28.5 ask=57
  LTC:USD    bid=53.31 ask=56.5
  XRP:USD    bid=1.35097 ask=1.35512
  XLM:USD    bid=0.145 ask=0.1742
  TRX:USD    bid=0.264243 ask=0.32
  ADA:USD    bid=0.254127 ask=0.285
  NEO:USD    bid=2.22 ask=4
  GAS:USD    bid=None ask=None
  BAT:USD    bid=None ask=None
  ATOM:USD   bid=1.76 ask=3
  XTZ:USD    bid=None ask=None
  ONT:USD    bid=None ask=None
  ONG:USD    bid=None ask=None
  USDT:USD   bid=None ask=None
  USDC:USD   bid=None ask=None
  LINK:USD   bid=8.652 ask=8.696
  MKR:USD    bid=None ask=None
  ZRX:USD    bid=None ask=None
  HOT:USD    bid=None ask=None
  DOT:USD    bid=0.8 ask=2.35
  COMP:USD   bid=14.66 ask=99.8
  ZIL:USD    bid=None ask=None
  UNI:USD    bid=None ask=None
  UMA:USD    bid=None ask=None
  SNX:USD    bid=None ask=None
  CRV:USD    bid=0.05 ask=0.52
  WBTC:USD   bid=None ask=None
  DAI:USD    bid=No

## 5. Order Book Endpoint (Primary Data Source)

Since tickers lack bid/ask sizes, the order book is our primary data source.

`GET /api/order_book/{symbol1}/{symbol2}/`

Response shape (from docs):
```json
{
  "timestamp": 1513173506,
  "timestamp_ms": 1513173506123,
  "bids": [[17670.3, 0.00250037]],
  "asks": [[17689.66, 0.01]],
  "pair": "BTC:USD",
  "id": 158217212,
  "sell_total": "1299.73578729",
  "buy_total": "10006393.37"
}
```

Key observations:
- `bids` and `asks` are **arrays-of-arrays** `[[price, amount], ...]` (same as Bitstamp)
- `timestamp_ms` provides millisecond precision (better than Bitstamp's microtimestamp)
- `pair` uses colon separator: `BTC:USD`
- Trailing slash may or may not be required (test both)

In [8]:
# Fetch BTC/USD order book
resp = httpx.get(f"{BASE_URL}/order_book/BTC/USD/")
print(f"Status: {resp.status_code}")
book = resp.json()

print(f"\nResponse fields: {list(book.keys())}")
print(f"Pair: {book.get('pair')}")
print(f"Timestamp: {book.get('timestamp')}")
print(f"Timestamp_ms: {book.get('timestamp_ms')}")
print(f"ID: {book.get('id')}")
print(f"Buy total: {book.get('buy_total')}")
print(f"Sell total: {book.get('sell_total')}")

# Examine top of book
print(f"\nTop bid: {book['bids'][0]}")
print(f"Top ask: {book['asks'][0]}")
print(f"Bid type: {type(book['bids'][0][0]).__name__}, {type(book['bids'][0][1]).__name__}")
print(f"Ask type: {type(book['asks'][0][0]).__name__}, {type(book['asks'][0][1]).__name__}")

# Check array depth
print(f"\nTotal bids: {len(book['bids'])}")
print(f"Total asks: {len(book['asks'])}")
print(f"First 3 bids: {book['bids'][:3]}")
print(f"First 3 asks: {book['asks'][:3]}")

Status: 200

Response fields: ['timestamp', 'timestamp_ms', 'bids', 'asks', 'pair', 'id', 'sell_total', 'buy_total']
Pair: BTC:USD
Timestamp: 1772914205
Timestamp_ms: 1772914205025
ID: 2403794768
Buy total: 536870.95
Sell total: 10.31350273

Top bid: [67208.5, 0.02]
Top ask: [67233.5, 0.005]
Bid type: float, float
Ask type: float, float

Total bids: 215
Total asks: 350
First 3 bids: [[67208.5, 0.02], [67196.0, 0.025], [67183.5, 0.025]]
First 3 asks: [[67233.5, 0.005], [67246.0, 0.025], [67258.5, 0.025]]


In [9]:
# Test trailing slash behavior
resp_with = httpx.get(f"{BASE_URL}/order_book/BTC/USD/")
resp_without = httpx.get(f"{BASE_URL}/order_book/BTC/USD")

print(f"With trailing slash:    status={resp_with.status_code}")
print(f"Without trailing slash: status={resp_without.status_code}")

# Check if responses are equivalent
if resp_with.status_code == 200 and resp_without.status_code == 200:
    book_with = resp_with.json()
    book_without = resp_without.json()
    print(f"Both have same pair: {book_with.get('pair')} == {book_without.get('pair')}")
    print("→ Trailing slash is optional (unlike Bitstamp which requires it)")

With trailing slash:    status=200
Without trailing slash: status=200
Both have same pair: BTC:USD == BTC:USD
→ Trailing slash is optional (unlike Bitstamp which requires it)


In [10]:
# Depth parameter test — can we limit the order book?
resp_depth = httpx.get(f"{BASE_URL}/order_book/BTC/USD/?depth=1")
print(f"depth=1 status: {resp_depth.status_code}")
if resp_depth.status_code == 200:
    book_depth = resp_depth.json()
    print(f"Bids returned: {len(book_depth['bids'])}")
    print(f"Asks returned: {len(book_depth['asks'])}")
    print(f"Top bid: {book_depth['bids'][0] if book_depth['bids'] else 'EMPTY'}")
    print(f"Top ask: {book_depth['asks'][0] if book_depth['asks'] else 'EMPTY'}")
    if len(book_depth["bids"]) == 1:
        print("→ depth=1 works! Can limit to top-of-book only (saves bandwidth)")
    else:
        print(f"→ depth=1 returned {len(book_depth['bids'])} bids — may not work as expected")

depth=1 status: 200
Bids returned: 1
Asks returned: 1
Top bid: [67208.5, 0.02]
Top ask: [67233.5, 0.005]
→ depth=1 works! Can limit to top-of-book only (saves bandwidth)


## 6. Parse into TopOfBook

Prototype the parser function that will become `connectors/cexio/parser.py`.

In [11]:
def parse_cexio_book(
    canonical_pair: str,
    book_data: dict,
    ts_local_ms: int,
) -> TopOfBook:
    """
    Parse CEX.IO order book response into TopOfBook.

    CEX.IO book format:
    {
        "timestamp": 1513173506,          # Unix seconds (integer)
        "timestamp_ms": 1513173506123,    # Unix milliseconds (integer) — if present
        "bids": [[price, amount], ...],   # arrays-of-arrays
        "asks": [[price, amount], ...],
        "pair": "BTC:USD",
        ...
    }
    """
    require_present(book_data.get("bids"), "bids")
    require_present(book_data.get("asks"), "asks")

    bids = book_data["bids"]
    asks = book_data["asks"]

    if not bids or not asks:
        raise ValueError(f"Empty order book for {canonical_pair}")

    top_bid = bids[0]  # [price, amount]
    top_ask = asks[0]  # [price, amount]

    # CEX.IO provides timestamp_ms when available, fall back to timestamp * 1000
    ts_exchange_ms: int
    if "timestamp_ms" in book_data:
        # timestamp_ms is an integer (not a string like Bitstamp's microtimestamp)
        ts_exchange_ms = int(book_data["timestamp_ms"])
    elif "timestamp" in book_data:
        ts_exchange_ms = int(book_data["timestamp"]) * 1000
    else:
        ts_exchange_ms = ts_local_ms  # fallback

    return tob_from_raw(
        venue="cexio",
        pair=canonical_pair,
        bid_px=str(top_bid[0]),
        bid_sz=str(top_bid[1]),
        ask_px=str(top_ask[0]),
        ask_sz=str(top_ask[1]),
        ts_exchange_ms=ts_exchange_ms,
        ts_local_ms=ts_local_ms,
    )


# Test with live data
ts_now = int(time.time() * 1000)
tob = parse_cexio_book("BTC/USD", book, ts_now)
print(f"TopOfBook for {tob.pair}:")
print(f"  venue:          {tob.venue!r}")
print(f"  bid_px:         {tob.bid_px!r}  type={type(tob.bid_px).__name__}")
print(f"  bid_sz:         {tob.bid_sz!r}  type={type(tob.bid_sz).__name__}")
print(f"  ask_px:         {tob.ask_px!r}  type={type(tob.ask_px).__name__}")
print(f"  ask_sz:         {tob.ask_sz!r}  type={type(tob.ask_sz).__name__}")
print(f"  ts_exchange_ms: {tob.ts_exchange_ms}")
print(f"  ts_local_ms:    {tob.ts_local_ms}")
print(f"  frozen:         {tob.__dataclass_params__.frozen}")

TopOfBook for BTC/USD:
  venue:          'cexio'
  bid_px:         Decimal('67208.5')  type=Decimal
  bid_sz:         Decimal('0.02')  type=Decimal
  ask_px:         Decimal('67233.5')  type=Decimal
  ask_sz:         Decimal('0.005')  type=Decimal
  ts_exchange_ms: 1772914205025
  ts_local_ms:    1772914205852
  frozen:         True


In [12]:
# Parse ALL available target pairs
print("Parsing all target pairs into TopOfBook:\n")
parsed_tobs: dict[str, TopOfBook] = {}
errors: dict[str, str] = {}

for canonical, cexio_sym in CEXIO_SYMBOL_MAP.items():
    try:
        parts = cexio_sym.split("/")
        resp = httpx.get(f"{BASE_URL}/order_book/{parts[0]}/{parts[1]}")
        time.sleep(2.5)  # Conservative: 300 req/10 min = ~2s between requests

        if resp.status_code != 200:
            errors[canonical] = f"HTTP {resp.status_code}: {resp.text[:100]}"
            continue

        data = resp.json()
        if "error" in data:
            errors[canonical] = f"API error: {data['error']}"
            continue

        ts_now = int(time.time() * 1000)
        tob = parse_cexio_book(canonical, data, ts_now)
        parsed_tobs[canonical] = tob
        print(
            f"  ✅ {canonical:10s}: bid={tob.bid_px} ({tob.bid_sz}), "
            f"ask={tob.ask_px} ({tob.ask_sz})"
        )
    except Exception as exc:
        errors[canonical] = str(exc)
        print(f"  ❌ {canonical:10s}: {exc}")

print(f"\nParsed: {len(parsed_tobs)}/{len(CEXIO_SYMBOL_MAP)} pairs")
if errors:
    print(f"Errors: {len(errors)}")
    for pair, err in errors.items():
        print(f"  {pair}: {err}")

Parsing all target pairs into TopOfBook:

  ✅ BTC/USD   : bid=67208.5 (0.02), ask=67233.5 (0.005)
  ✅ LTC/USD   : bid=53.31 (15.0), ask=56.5 (5.51206408)
  ❌ LTC/USDC  : Required value 'bids' is missing
  ❌ LTC/BTC   : Required value 'bids' is missing
  ✅ SOL/USD   : bid=82.6558 (2.0), ask=82.85 (10.0)
  ✅ SOL/USDC  : bid=65.0 (4.603874), ask=125.0 (1.0)

Parsed: 4/8 pairs
Errors: 4
  BTC/USDC: API error: Invalid Symbols Pair
  LTC/USDC: Required value 'bids' is missing
  LTC/BTC: Required value 'bids' is missing
  SOL/BTC: API error: Invalid Symbols Pair


## 7. Async httpx Pattern

Production connector will use `BaseAsyncConnector._fetch_tickers_per_pair()` template.
CEX.IO has no batch order book endpoint, so we fetch each pair individually (same as
Gemini, Coinbase, Bitstamp).

**Note**: Using sync httpx here due to Python 3.14 + nest_asyncio + anyio compatibility
issue (LL-050/LL-064). Production connector runs in a proper event loop outside Jupyter.

In [13]:
# Async code reference for production connector (DO NOT RUN in Jupyter 3.14)
#
# async def fetch_book_async(
#     client: httpx.AsyncClient,
#     symbol1: str,
#     symbol2: str,
# ) -> dict:
#     resp = await client.get(f"/order_book/{symbol1}/{symbol2}")
#     resp.raise_for_status()
#     return resp.json()
#
# Production connector: BaseAsyncConnector._fetch_tickers_per_pair()
# will iterate over pairs with rate limiting between each request.

# --- Sync simulation of production polling ---
print("Sequential fetch with 2.5s delay (simulates production polling):\n")
start = time.time()

tobs = {}
with httpx.Client(base_url=BASE_URL, timeout=10.0) as client:
    for canonical, cexio_sym in CEXIO_SYMBOL_MAP.items():
        try:
            parts = cexio_sym.split("/")
            resp = client.get(f"/order_book/{parts[0]}/{parts[1]}")
            resp.raise_for_status()
            data = resp.json()
            ts_now = int(time.time() * 1000)

            tob = parse_cexio_book(canonical, data, ts_now)
            tobs[canonical] = tob
            print(
                f"  ✅ {canonical:10s}: bid={tob.bid_px} ({tob.bid_sz}), "
                f"ask={tob.ask_px} ({tob.ask_sz})"
            )
        except Exception as exc:
            print(f"  ❌ {canonical:10s}: {exc}")

        time.sleep(2.5)

elapsed = time.time() - start
print(f"\nFetched {len(tobs)} pairs in {elapsed:.2f}s")
print(f"Avg per pair: {elapsed / len(CEXIO_SYMBOL_MAP):.2f}s (includes 2.5s delay)")
print("\n→ Production connector will use BaseAsyncConnector._fetch_tickers_per_pair()")
print("→ Async httpx not tested here due to Python 3.14 + nest_asyncio issue (LL-050)")

Sequential fetch with 2.5s delay (simulates production polling):

  ✅ BTC/USD   : bid=67208.5 (0.02), ask=67233.5 (0.005)
  ❌ BTC/USDC  : Required value 'bids' is missing
  ✅ LTC/USD   : bid=53.31 (15.0), ask=56.5 (5.51206408)
  ❌ LTC/USDC  : Required value 'bids' is missing
  ❌ LTC/BTC   : Required value 'bids' is missing
  ✅ SOL/USD   : bid=82.6658 (2.0), ask=82.85 (10.0)
  ✅ SOL/USDC  : bid=65.0 (4.603874), ask=125.0 (1.0)
  ❌ SOL/BTC   : Required value 'bids' is missing

Fetched 4 pairs in 21.32s
Avg per pair: 2.67s (includes 2.5s delay)

→ Production connector will use BaseAsyncConnector._fetch_tickers_per_pair()
→ Async httpx not tested here due to Python 3.14 + nest_asyncio issue (LL-050)


## 8. Rate Limit Testing

CEX.IO documented limits:
- **Public REST**: 300 requests per 10 minutes (~30 req/min, ~0.5 req/sec)
- **Private REST**: 600 requests per 10 minutes

This is significantly more conservative than other exchanges:
- Kraken: 1 req/sec (public)
- Coinbase: 10 req/sec (public)
- Bitstamp: 400 req/sec (public)
- Gemini: 120 req/min (public)

For 8 pairs at 2s intervals = 16s per scan cycle. This is within the 5s polling interval
if we can go faster, but may need a longer interval.

Let's test the actual behavior.

In [14]:
# Rate limit test: burst of 10 rapid requests
print("Rate limit test: 10 rapid requests to /api/ticker/BTC/USD\n")

results = []
start = time.time()
for i in range(10):
    t0 = time.time()
    resp = httpx.get(f"{BASE_URL}/ticker/BTC/USD")
    elapsed = time.time() - t0
    results.append(
        {
            "index": i + 1,
            "status": resp.status_code,
            "elapsed_ms": round(elapsed * 1000),
        }
    )
    print(f"  #{i + 1:2d}: status={resp.status_code} elapsed={elapsed * 1000:.0f}ms")

total = time.time() - start
rate_429 = sum(1 for r in results if r["status"] == 429)
print(f"\nTotal time: {total:.2f}s")
print(f"Avg per request: {total / 10 * 1000:.0f}ms")
print(f"429 rate limits hit: {rate_429}/10")
if rate_429 > 0:
    print("⚠️ Rate limiting is enforced on burst traffic!")
else:
    print("✅ No 429s on burst — but respect documented 300/10min limit")

Rate limit test: 10 rapid requests to /api/ticker/BTC/USD

  # 1: status=200 elapsed=240ms
  # 2: status=200 elapsed=222ms
  # 3: status=200 elapsed=250ms
  # 4: status=200 elapsed=243ms
  # 5: status=200 elapsed=219ms
  # 6: status=200 elapsed=245ms
  # 7: status=200 elapsed=229ms
  # 8: status=200 elapsed=264ms
  # 9: status=200 elapsed=252ms
  #10: status=200 elapsed=270ms

Total time: 2.43s
Avg per request: 243ms
429 rate limits hit: 0/10
✅ No 429s on burst — but respect documented 300/10min limit


In [15]:
# Sustained rate test: 15 requests at ~2s intervals (simulating production)
print("Sustained rate test: 15 requests at ~2s intervals\n")

results2 = []
start = time.time()
for i in range(15):
    t0 = time.time()
    resp = httpx.get(f"{BASE_URL}/ticker/BTC/USD")
    elapsed = time.time() - t0
    results2.append({"status": resp.status_code, "elapsed_ms": round(elapsed * 1000)})
    print(f"  #{i + 1:2d}: status={resp.status_code} elapsed={elapsed * 1000:.0f}ms")
    time.sleep(2.0)

total = time.time() - start
rate_429 = sum(1 for r in results2 if r["status"] == 429)
print(f"\nTotal time: {total:.2f}s")
print(f"429 rate limits hit: {rate_429}/15")

# Recommend rate limiter interval
print("\n→ Recommended RateLimiter interval: 2000ms (conservative)")
print("→ 8 pairs × 2s = 16s per scan cycle")
print("→ Need polling interval ≥ 20s for CEX.IO (vs 5s for other exchanges)")
print("→ OR: run CEX.IO on a separate, slower polling cadence")

Sustained rate test: 15 requests at ~2s intervals

  # 1: status=200 elapsed=237ms
  # 2: status=200 elapsed=224ms
  # 3: status=200 elapsed=240ms
  # 4: status=200 elapsed=257ms
  # 5: status=200 elapsed=244ms
  # 6: status=200 elapsed=229ms
  # 7: status=200 elapsed=253ms
  # 8: status=200 elapsed=264ms
  # 9: status=200 elapsed=254ms
  #10: status=200 elapsed=222ms
  #11: status=200 elapsed=317ms
  #12: status=200 elapsed=236ms
  #13: status=200 elapsed=296ms
  #14: status=200 elapsed=239ms
  #15: status=200 elapsed=228ms

Total time: 33.81s
429 rate limits hit: 0/15

→ Recommended RateLimiter interval: 2000ms (conservative)
→ 8 pairs × 2s = 16s per scan cycle
→ Need polling interval ≥ 20s for CEX.IO (vs 5s for other exchanges)
→ OR: run CEX.IO on a separate, slower polling cadence


## 9. Error Handling

Test various error scenarios to document error response format.

In [16]:
# Test 1: Invalid symbol pair
print("Test 1: Invalid symbol pair")
resp = httpx.get(f"{BASE_URL}/order_book/INVALID/PAIR")
print(f"  Status: {resp.status_code}")
print(f"  Content-Type: {resp.headers.get('content-type', 'N/A')}")
print(f"  Body: {resp.text[:200]}")

print()
time.sleep(2.5)

# Test 2: Valid symbols but unlisted pair
print("Test 2: Potentially unlisted pair (DOGE/EUR)")
resp = httpx.get(f"{BASE_URL}/order_book/DOGE/EUR")
print(f"  Status: {resp.status_code}")
print(f"  Content-Type: {resp.headers.get('content-type', 'N/A')}")
print(f"  Body: {resp.text[:200]}")

print()
time.sleep(2.5)

# Test 3: Wrong endpoint path
print("Test 3: Wrong endpoint")
resp = httpx.get(f"{BASE_URL}/nonexistent/BTC/USD")
print(f"  Status: {resp.status_code}")
print(f"  Content-Type: {resp.headers.get('content-type', 'N/A')}")
print(f"  Body (first 200 chars): {resp.text[:200]}")

print()
time.sleep(2.5)

# Test 4: Lowercase symbols
print("Test 4: Lowercase symbols (btc/usd)")
resp = httpx.get(f"{BASE_URL}/order_book/btc/usd")
print(f"  Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(f"  Pair returned: {data.get('pair', 'N/A')}")
    print("  → CEX.IO may be case-insensitive (check)")
else:
    print(f"  Body: {resp.text[:200]}")
    print("  → CEX.IO is case-sensitive — must use uppercase")

Test 1: Invalid symbol pair
  Status: 200
  Content-Type: text/json
  Body: {"error":"Invalid Symbols Pair"}

Test 2: Potentially unlisted pair (DOGE/EUR)
  Status: 200
  Content-Type: text/json
  Body: {"error":"Invalid Symbols Pair"}

Test 3: Wrong endpoint
  Status: 404
  Content-Type: text/html
  Body (first 200 chars): <!DOCTYPE html>
<html>
	<head>
		<title>404 - CEX.IO</title>
		<meta charset="utf-8">
		<meta name="viewport" content="width=device-width, initial-scale=1.0">

		<link rel="shortcut icon" href="/favic

Test 4: Lowercase symbols (btc/usd)
  Status: 200
  Pair returned: N/A
  → CEX.IO may be case-insensitive (check)


## 10. Symbol Details & Trading Precision

Use the `/api/currency_limits` endpoint to get precision, min/max order sizes.

In [17]:
# Fetch full currency limits
resp = httpx.post(f"{BASE_URL}/currency_limits")
limits = resp.json()

if limits.get("ok") == "ok":
    print("Trading precision for target pairs:\n")
    print(
        f"{'Pair':12s} {'pricePrecision':>15s} {'minLotSize':>12s} "
        f"{'minLotSizeS2':>13s} {'maxLotSize':>11s}"
    )
    print("-" * 70)

    for pair_info in limits["data"]["pairs"]:
        canonical = f"{pair_info['symbol1']}/{pair_info['symbol2']}"
        if canonical in CEXIO_SYMBOL_MAP:
            print(
                f"{canonical:12s} "
                f"{pair_info.get('pricePrecision', 'N/A'):>15} "
                f"{pair_info.get('minLotSize', 'N/A'):>12} "
                f"{pair_info.get('minLotSizeS2', 'N/A'):>13} "
                f"{pair_info.get('maxLotSize', 'N/A'):>11}"
            )

    # USD vs USDC comparison
    print("\nUSD vs USDC pair comparison:")
    for pair_info in limits["data"]["pairs"]:
        canonical = f"{pair_info['symbol1']}/{pair_info['symbol2']}"
        if canonical in ("BTC/USD", "BTC/USDC"):
            print(
                f"  {canonical}: precision={pair_info.get('pricePrecision')} "
                f"minLot={pair_info.get('minLotSize')}"
            )
else:
    print(f"Error: {limits}")

Trading precision for target pairs:

Pair          pricePrecision   minLotSize  minLotSizeS2  maxLotSize
----------------------------------------------------------------------


TypeError: unsupported format string passed to NoneType.__format__

## 11. Fee Schedule

### Trading Fees (from CEX.IO fee schedule page + research)

| 30-Day Volume | Taker | Maker |
|---------------|-------|-------|
| < $10,000 | 0.25% | 0.15% |
| $10K – $100K | 0.23% | 0.13% |
| $100K – $250K | 0.21% | 0.12% |
| $250K – $500K | 0.19% | 0.11% |
| $500K – $1M | 0.17% | 0.10% |
| $1M – $2.5M | 0.15% | 0.09% |
| $2.5M – $5M | 0.13% | 0.08% |
| ... | ... | ... |
| $5B+ | 0.01% | 0.00% |

**For our system**: Use base tier 0.25% taker (conservative, per DEC-013 flat fee model).

### Withdrawal Fees
- BTC: 0.0005 BTC (documented)
- LTC, SOL, USD, USDC: Need to verify via CEX.IO fee page
- Note: Crypto withdrawal fees = on-chain network fees, which vary

### Comparison with Other Exchanges
| Exchange | Taker Fee |
|----------|-----------|
| Kraken | 0.26% |
| Coinbase | 0.60% |
| Gemini | 0.40% |
| Bitstamp | 0.40% |
| OKX | 0.10% |
| **CEX.IO** | **0.25%** |

CEX.IO's taker fee is very competitive — second lowest after OKX among our exchanges.

## 12. Timestamp Format Deep Dive

CEX.IO provides both seconds and milliseconds timestamps in order book responses.

In [ ]:
# Examine timestamp formats across endpoints
from datetime import UTC, datetime

# Order book timestamps
resp = httpx.get(f"{BASE_URL}/order_book/BTC/USD")
book = resp.json()
ts = book.get("timestamp")
ts_ms = book.get("timestamp_ms")

print("Order Book timestamps:")
print(f"  timestamp:    {ts} (type={type(ts).__name__})")
print(f"  timestamp_ms: {ts_ms} (type={type(ts_ms).__name__})")

if ts_ms:
    dt = datetime.fromtimestamp(ts_ms / 1000, tz=UTC)
    print(f"  Decoded:      {dt.isoformat()}")
    print("  ms precision: ✅ (integer, not string)")

time.sleep(2.5)

# Ticker timestamps
resp = httpx.get(f"{BASE_URL}/ticker/BTC/USD")
ticker = resp.json()
ts_ticker = ticker.get("timestamp")

print("\nTicker timestamps:")
print(f"  timestamp: {ts_ticker} (type={type(ts_ticker).__name__})")
if ts_ticker:
    dt = datetime.fromtimestamp(int(ts_ticker), tz=UTC)
    print(f"  Decoded:   {dt.isoformat()}")
    print("  Note: ticker timestamp is a STRING, not integer")

print("\n→ Production connector should use timestamp_ms from order book (integer)")
print("→ No conversion needed — already in our expected ms format")

# Compare timestamp formats across exchanges
print("\nTimestamp format comparison:")
print("  Kraken:   Unix seconds (string in ticker, not in book)")
print("  Coinbase: ISO 8601 with microseconds")
print("  Gemini:   Unix seconds (integer string in book entries)")
print("  Bitstamp: Unix seconds + microtimestamp (string)")
print("  OKX:      Unix milliseconds (string)")
print("  CEX.IO:   Unix seconds (int) + timestamp_ms (int) ← EASIEST")

## 13. Summary & Production Connector Design Notes

### Ohio Eligibility
✅ **CONFIRMED** — CEX.IO holds Ohio MTL OHMT176

### Pair Availability
(To be filled after running notebook — expecting most/all 8 pairs)

### Key API Facts

| Property | Value |
|----------|-------|
| Base URL | `https://cex.io/api` |
| Data source | Order book (`/api/order_book/{sym1}/{sym2}`) |
| Symbol format | URL path segments: `/BTC/USD` |
| Response pair format | Colon-separated: `BTC:USD` |
| Book entry format | Arrays-of-arrays: `[[price, amount], ...]` |
| Timestamp | `timestamp_ms` (Unix ms, integer) |
| Rate limit | 300 req/10 min (public) |
| Taker fee (base) | 0.25% |
| Auth needed | No (Phase 1 public endpoints) |
| Batch order book | ❌ No — per-pair only |
| Depth parameter | Test result TBD |
| Case sensitivity | Test result TBD |

### Connector Design

```
connectors/cexio/
    __init__.py
    symbols.py      # CEXIO_SYMBOL_MAP + create_translator()
    parser.py       # parse_cexio_book() → TopOfBook
    client.py       # CexioClient(BaseAsyncConnector)
```

**Pattern**: Same as Gemini/Bitstamp — per-pair order book fetches via `_fetch_tickers_per_pair()`.

### Key Differences from Other Connectors

| Aspect | CEX.IO | Others |
|--------|--------|--------|
| Symbol in URL | Path segments `/BTC/USD` | Single string param |
| Book format | `[[price, amt]]` | Same as Bitstamp |
| Timestamp | `timestamp_ms` (int) | Varies by exchange |
| Rate limit | 2s between requests (conservative) | 100-500ms |
| Trailing slash | Optional | Bitstamp requires it |

### Rate Limit Implications
CEX.IO's 300 req/10 min limit is the most restrictive among our exchanges.
With 8 pairs at 2s intervals = 16s per scan cycle. Options:
1. Increase polling interval to 20s for CEX.IO specifically
2. Run CEX.IO on a separate polling cadence
3. Accept that CEX.IO data will be slightly staler than other exchanges

### Production Recommendations
1. Use `BaseAsyncConnector` with `_fetch_tickers_per_pair()` template
2. RateLimiter interval: **2000ms** (conservative for 300/10min)
3. Order book endpoint (NOT ticker) — tickers lack bid/ask sizes
4. `timestamp_ms` field is ideal — integer milliseconds, no conversion needed
5. Handle both JSON errors and potential HTML error pages (test results TBD)
6. Venue name: `"cexio"` (no dots in identifier, per module naming convention)

### Fixture Generation
Save representative order book responses for test fixtures:
- `tests/fixtures/cexio_book_btc_usd.json`
- `tests/fixtures/cexio_book_ltc_btc.json`
- `tests/fixtures/cexio_book_sol_usdc.json`